# 04 - Training YOLOv8n + CBAM (Proposed Method)

Notebook ini melatih YOLOv8n yang dimodifikasi dengan CBAM (Convolutional Block Attention Module).
CBAM disisipkan di dua posisi: setelah C2f block di backbone, dan di neck (FPN).
Urutan: setup -> definisi CBAM -> modifikasi arsitektur -> training -> evaluasi -> simpan hasil.

## Setup Environment

In [1]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    PROJECT_ROOT = '/content/drive/MyDrive/malaria-detection'
    os.chdir(PROJECT_ROOT)
else:
    import os
    # gunakan path absolut langsung, jangan pakai os.path.abspath('..')
    # karena tergantung dari mana notebook dijalankan
    PROJECT_ROOT = r'D:\malaria-detection'
    os.chdir(PROJECT_ROOT)

print('working directory:', os.getcwd())
print('PROJECT_ROOT:', PROJECT_ROOT)

working directory: D:\malaria-detection
PROJECT_ROOT: D:\malaria-detection


In [2]:
!pip install -q ultralytics


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json
import time
import cv2
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.block import C2f
from ultralytics.nn.modules.conv import Conv

In [4]:
# cek gpu
print('torch version :', torch.__version__)
print('cuda available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('gpu           :', torch.cuda.get_device_name(0))
    DEVICE = 0
else:
    print('gpu tidak ditemukan, training pakai cpu')
    DEVICE = 'cpu'

torch version : 2.5.1+cu121
cuda available: True
gpu           : NVIDIA GeForce RTX 3060 Laptop GPU


In [5]:
# load summary dari notebook 01
summary_path = os.path.join(PROJECT_ROOT, 'notebook01_summary.json')

if not os.path.exists(summary_path):
    raise FileNotFoundError('notebook01_summary.json tidak ditemukan, pastikan notebook 01 sudah dijalankan')

with open(summary_path) as f:
    summary = json.load(f)

print('status notebook 01:', summary['status'])
print('train            :', summary['total_train'], 'gambar')
print('val              :', summary['total_val'], 'gambar')
print('test             :', summary['total_test'], 'gambar')

YAML_PATH = summary['yaml_path']

status notebook 01: done
train            : 21676 gambar
val              : 5606 gambar
test             : 2803 gambar


## Definisi Modul CBAM

In [6]:
class ChannelAttention(nn.Module):
    # squeeze seluruh spatial dimension lalu pelajari bobot tiap channel
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        out     = self.sigmoid(avg_out + max_out)
        return x * out.view(x.size(0), x.size(1), 1, 1)


class SpatialAttention(nn.Module):
    # compress channel dimension lalu pelajari bobot tiap lokasi spatial
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * out


class CBAM(nn.Module):
    # channel attention diikuti spatial attention
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


print('modul CBAM didefinisikan')

# test forward pass dengan dummy input
dummy = torch.randn(1, 64, 20, 20)
cbam  = CBAM(64)
out   = cbam(dummy)
print(f'test forward pass: input {dummy.shape} -> output {out.shape}')

modul CBAM didefinisikan
test forward pass: input torch.Size([1, 64, 20, 20]) -> output torch.Size([1, 64, 20, 20])


## Modifikasi Arsitektur YOLOv8n

In [7]:
class C2f_CBAM(nn.Module):
    # wrapper C2f + CBAM yang mewarisi semua atribut penting dari C2f asli
    def __init__(self, c2f_module, channels):
        super().__init__()
        self.c2f  = c2f_module
        self.cbam = CBAM(channels)

        # salin semua atribut routing ultralytics dari c2f asli
        # atribut ini dibutuhkan saat forward pass di _predict_once
        for attr in ['f', 'i', 'type', 'np']:
            if hasattr(c2f_module, attr):
                setattr(self, attr, getattr(c2f_module, attr))

    def forward(self, x):
        return self.cbam(self.c2f(x))


def insert_cbam(model):
    target_layers = [2, 4, 6, 8, 12, 15, 18, 21]
    channel_map = {
        2  : 32,
        4  : 64,
        6  : 128,
        8  : 256,
        12 : 128,
        15 : 64,
        18 : 128,
        21 : 256,
    }

    modified = 0
    for idx, layer in enumerate(model.model.model):
        if idx in target_layers and isinstance(layer, C2f):
            channels = channel_map[idx]
            new_layer = C2f_CBAM(layer, channels)
            model.model.model[idx] = new_layer
            modified += 1
            print(f'  layer {idx:2d} (ch={channels}) -> C2f_CBAM')

    print(f'\ntotal layer dimodifikasi: {modified}')
    return model


print('definisi ulang selesai')

definisi ulang selesai


In [8]:
# load ulang model dari awal
model = YOLO('yolov8n.pt')

print('menyisipkan CBAM...')
model = insert_cbam(model)

total_params = sum(p.numel() for p in model.model.parameters())
print(f'total parameter: {total_params:,}')

menyisipkan CBAM...
  layer  2 (ch=32) -> C2f_CBAM
  layer  4 (ch=64) -> C2f_CBAM
  layer  6 (ch=128) -> C2f_CBAM
  layer  8 (ch=256) -> C2f_CBAM
  layer 12 (ch=128) -> C2f_CBAM
  layer 15 (ch=64) -> C2f_CBAM
  layer 18 (ch=128) -> C2f_CBAM
  layer 21 (ch=256) -> C2f_CBAM

total layer dimodifikasi: 8
total parameter: 3,181,664


In [9]:
# verifikasi cbam terpasang dengan forward pass dummy
dummy_img = torch.randn(1, 3, 640, 640)
if torch.cuda.is_available():
    dummy_img  = dummy_img.cuda()
    model.model = model.model.cuda()

with torch.no_grad():
    _ = model.model(dummy_img)

print('forward pass berhasil, CBAM terpasang dengan benar')

forward pass berhasil, CBAM terpasang dengan benar


## Training YOLOv8n + CBAM

In [10]:
SAVE_DIR = os.path.join(PROJECT_ROOT, 'runs', 'yolov8n-cbam')
os.makedirs(SAVE_DIR, exist_ok=True)

print('hasil training akan disimpan di:', SAVE_DIR)

hasil training akan disimpan di: D:\malaria-detection\runs\yolov8n-cbam


In [11]:
# training dengan hyperparameter yang sama persis dengan notebook 02 dan 03
results = model.train(
    data        = YAML_PATH,
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    optimizer   = 'AdamW',
    lr0         = 0.001,
    patience    = 20,
    device      = DEVICE,
    project     = SAVE_DIR,
    name        = 'train',
    save        = True,
    save_period = 10,
    exist_ok    = True,
    verbose     = True,
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\malaria-detection\malaria.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, 

## Evaluasi pada Test Set

In [13]:
# cara yang benar: load best.pt langsung pakai YOLO
# ultralytics menyimpan arsitektur + weights sekaligus di dalam best.pt
# tidak perlu rebuild model dari scratch

best_weights = os.path.join(SAVE_DIR, 'train', 'weights', 'best.pt')

if not os.path.exists(best_weights):
    raise FileNotFoundError('best.pt tidak ditemukan, pastikan training sudah selesai')

# load langsung — ultralytics akan handle arsitektur otomatis
model_best = YOLO(best_weights)

print('best weights loaded dari:', best_weights)
print('jumlah parameter:', sum(p.numel() for p in model_best.model.parameters()))

best weights loaded dari: D:\malaria-detection\runs\yolov8n-cbam\train\weights\best.pt
jumlah parameter: 3011238


In [14]:
# evaluasi di test set
metrics = model_best.val(
    data   = YAML_PATH,
    split  = 'test',
    device = DEVICE,
)

print('\nhasil evaluasi YOLOv8n+CBAM di test set:')
print(f'  mAP@0.5        : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95   : {metrics.box.map:.4f}')
print(f'  Precision      : {metrics.box.p[0]:.4f}')
print(f'  Recall         : {metrics.box.r[0]:.4f}')

Ultralytics 8.4.56  Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 4.83.0 MB/s, size: 54.6 KB)
val: Scanning D:\malaria-detection\datasets\processed\labels\test.cache... 2803 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2803/2803  0.0s
val: D:\malaria-detection\datasets\processed\images\test\mpidb_0235.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 176/176 8.3it/s 21.2s0.1ss
                   all       2803       3057       0.94      0.938       0.96      0.934
           parasitized       1406       1660      0.938      0.883       0.93      0.879
            uninfected       1397       1397      0.943      0.994       0.99       0.99
Speed: 1.3ms preprocess, 2.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results

In [15]:
# ukur fps rata-rata di test set
test_imgs = list(Path(summary['PATHS']['img_test']).glob('*.jpg'))[:100]

times = []
for img_path in test_imgs:
    img = cv2.imread(str(img_path))
    t0  = time.time()
    model_best.predict(img, verbose=False, device=DEVICE)
    times.append(time.time() - t0)

avg_ms  = (sum(times) / len(times)) * 1000
avg_fps = 1 / (sum(times) / len(times))

print(f'rata-rata inferensi : {avg_ms:.2f} ms')
print(f'rata-rata FPS       : {avg_fps:.1f}')

rata-rata inferensi : 10.64 ms
rata-rata FPS       : 94.0


## Simpan Hasil

In [16]:
# hitung jumlah parameter cbam yang ditambahkan
total_params = sum(p.numel() for p in model_best.model.parameters())
base_params  = 3_011_238  # parameter yolov8n vanilla (dari notebook 02)
cbam_params  = total_params - base_params

print(f'total parameter yolov8n+cbam : {total_params:,}')
print(f'parameter yolov8n vanilla    : {base_params:,}')
print(f'parameter tambahan dari CBAM : {cbam_params:,}')

total parameter yolov8n+cbam : 3,006,038
parameter yolov8n vanilla    : 3,011,238
parameter tambahan dari CBAM : -5,200


In [19]:
total_params = sum(p.numel() for p in model_best.model.parameters())

precision = float(metrics.box.p[0])
recall    = float(metrics.box.r[0])
f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

nb04_results = {
    'model'         : 'YOLOv8n+CBAM',
    'best_weights'  : best_weights,
    'map50'         : round(metrics.box.map50, 4),
    'map50_95'      : round(metrics.box.map, 4),
    'precision'     : round(precision, 4),
    'recall'        : round(recall, 4),
    'f1'            : round(f1, 4),
    'fps'           : round(avg_fps, 2),
    'avg_ms'        : round(avg_ms, 2),
    'total_params'  : total_params,
    'cbam_positions': ['backbone (C2f layer 2,4,6,8)', 'neck (C2f layer 12,15,18,21)'],
}

result_path = os.path.join(PROJECT_ROOT, 'notebook04_results.json')
with open(result_path, 'w') as f:
    json.dump(nb04_results, f, indent=2)

print('hasil disimpan di:', result_path)
print(json.dumps(nb04_results, indent=2))

hasil disimpan di: D:\malaria-detection\notebook04_results.json
{
  "model": "YOLOv8n+CBAM",
  "best_weights": "D:\\malaria-detection\\runs\\yolov8n-cbam\\train\\weights\\best.pt",
  "map50": 0.96,
  "map50_95": 0.9343,
  "precision": 0.9381,
  "recall": 0.8825,
  "f1": 0.9095,
  "fps": 93.96,
  "avg_ms": 10.64,
  "total_params": 3006038,
  "cbam_positions": [
    "backbone (C2f layer 2,4,6,8)",
    "neck (C2f layer 12,15,18,21)"
  ]
}


In [20]:
print('notebook 04 selesai')
print('best weights:', best_weights)
print()
print('lanjut ke notebook 05_evaluation_comparison.ipynb')

notebook 04 selesai
best weights: D:\malaria-detection\runs\yolov8n-cbam\train\weights\best.pt

lanjut ke notebook 05_evaluation_comparison.ipynb
